# 02 · 작업 미완료 감지 — 데이터 적합성 검토

**현업 가설:** 로봇이 동작을 이어가도 실제 물체가 목표 상태에 도달하지 않을 수 있다. 영상과 행동·상태를 연결해 담당자가 확인할 근거를 제공한다.

21개 샘플의 잠정 시각 검토입니다. 모델의 예측 결과나 전체 데이터의 실패율이 아닙니다. 실패 후보는 연속 프레임으로 추가 검토했으며, 모든 라벨은 사용자의 확인이 필요합니다.

이 노트북을 실행하면 실제 장면이 출력에 저장됩니다. 출력 포함 버전은 `reports/local/`에 보관하세요.


In [ ]:
from pathlib import Path
import json
import runpy
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "scripts/audit_failure_samples.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("리포지토리 안에서 실행하세요.")
# 동일 시드로 프레임과 점검용 manifest를 생성합니다. 원본은 수정하지 않습니다.
runpy.run_path(str(ROOT / "scripts/audit_failure_samples.py"), run_name="__main__")
AUDIT = ROOT / "reports/local/failure-audit"
manifest = json.loads((AUDIT / "manifest.json").read_text())
by_id = {row["sample_id"]: row for row in manifest}
assert all(row["video_frames"] == row["parquet_rows"] == row["metadata_frames"] for row in manifest)
print("21개 샘플의 MP4 프레임 수 / Parquet 행 수 / 메타데이터 길이 일치")


## 1. 판정 기준과 검토 결과

완료 여부와 도중에 발생한 사건은 분리합니다. 작업 지시 자체에 낙하가 포함되면 '낙하=실패'로 처리하지 않습니다.

`apparent_success`: 목표 상태가 보이는 잠정 완료, `failure_candidate`: 목표 미달성의 시각 근거가 있는 후보, `undetermined`: 증거 부족.

G01~G04는 집기·배치, G05는 의도적 낙하/재집기, G06은 쌓기 실패 후보, G07은 같은 환경의 완료 비교 사례입니다.


In [ ]:
annotations = {}
for group in range(1, 5):
    for sample in range(1, 4):
        annotations[f"G{group:02d}-S{sample:02d}"] = {
            "outcome": "apparent_success", "event": "no_failure_observed_in_sampled_frames",
            "note": "균등 12프레임에서 물체의 이송과 용기 안 종결 상태가 보임. 짧은 중간 사건은 미검증.",
            "review_frames": None,
        }
for sample in range(1, 4):
    annotations[f"G05-S{sample:02d}"] = {
        "outcome": "undetermined", "event": "intentional_drop_task",
        "note": "떨어뜨린 뒤 다시 집는 작업. 가림/흐림으로 전체 완료 여부 유보. 일반 낙하 실패로 라벨링하지 않음.",
        "review_frames": None,
    }
annotations["G06-S01"] = {"outcome": "failure_candidate", "event": "grasp_not_established", "note": "28~30프레임에서 큐브가 밀리고 37~43프레임에서 빈 그리퍼가 다음 위치로 이동. 종결 장면에서도 쌓이지 않음.", "review_frames": [28, 43]}
annotations["G06-S02"] = {"outcome": "failure_candidate", "event": "grasp_not_established", "note": "24~35프레임 집기 시도 이후 36~43프레임에서 큐브를 남겨 두고 이동. 종결 장면에서도 쌓이지 않음.", "review_frames": [24, 43]}
annotations["G06-S03"] = {"outcome": "failure_candidate", "event": "object_separation_before_goal", "note": "27~32프레임에서 큐브가 들리고 33~34프레임에 그리퍼에서 이탈하여 초록 큐브 옆에 남음. 정확한 기계적 원인은 미확정.", "review_frames": [27, 34]}
for sample in range(1, 4):
    annotations[f"G07-S{sample:02d}"] = {
        "outcome": "apparent_success", "event": "stack_completed",
        "note": "종결 장면에서 그리퍼가 분리되고 빨간 큐브가 초록 큐브 위에 남음.",
        "review_frames": None,
    }
review = pd.DataFrame([{"sample_id": row["sample_id"], "dataset": row["dataset"], "episode": row["episode"], **annotations[row["sample_id"]]} for row in manifest])
display(review[["sample_id", "outcome", "event"]])
display(review.groupby(["outcome", "event"]).size().rename("reviewed_samples").to_frame())
review.to_json(AUDIT / "review_annotations.json", orient="records", force_ascii=False, indent=2)
print("에이전트의 잠정 판정: 완료 15 / 실패 후보 3 / 판단 유보 3. 현업 실패율이나 모델 성능이 아닙니다.")


## 2. 실제 장면 — 21개 샘플

각 이미지는 왼쪽 위부터 시간순이며 프레임 번호가 표시됩니다. 검토 근거를 먼저 보고 라벨을 재검토하세요.
원본 task 문장은 수집 조건을 이해하기 위한 표시입니다. 오류 정보가 포함되어 있으므로 모델 입력으로 사용하면 안 됩니다.


In [ ]:
for row in manifest:
    sid = row["sample_id"]
    note = annotations[sid]
    display(Markdown(f"### {sid} · {note['outcome']}\n\n{row['dataset']} / {row['episode']}\n\n원본 지시문: {'; '.join(row['tasks'])}\n\n검토: {note['note']}"))
    display(Image(filename=str(AUDIT / f"{sid}.jpg")))


## 3. 실패 후보의 연속 프레임

이 샘플들에는 20~43번 프레임을 빠짐없이 표시합니다. G06-S03의 32→33→34번 프레임에서 들린 물체가 목표 밖에 남는 장면을 확인할 수 있습니다.


In [ ]:
for sid in ["G06-S01", "G06-S02", "G06-S03"]:
    display(Markdown(f"### {sid}\n\n{annotations[sid]['note']}"))
    for start in [20, 32]:
        display(Image(filename=str(AUDIT / f"{sid}-detail-{start}.jpg")))


## 4. 영상 시점과 그리퍼 행동·상태를 연결하기

회색 구간은 **사후 검토한 증거 구간**이며 실시간 탐지 결과가 아닙니다. 원본 수치의 물리 단위를 가정하지 않습니다. 명령과 상태가 비슷하게 움직여도 물체를 실제 잡았다는 증거는 되지 않습니다.


In [ ]:
ids = ["G06-S01", "G06-S02", "G06-S03", "G07-S01", "G07-S02", "G07-S03"]
fig, axes = plt.subplots(3, 2, figsize=(12, 9))
offsets = []
for ax, sid in zip(axes.flat, ids):
    row = by_id[sid]
    df = pq.read_table(ROOT / row["parquet"]).to_pandas()
    actions = np.stack(df["action"])
    states = np.stack(df["observation.state"])
    t = df["timestamp"].to_numpy()
    video_t = np.array(row["timestamps"], dtype=float)
    offsets.append({"sample_id": sid, "max_timestamp_difference_seconds": float(np.max(np.abs(t - video_t)))})
    ax.plot(t, actions[:, 5], label="Gripper command")
    ax.plot(t, states[:, 5], label="Observed gripper state", linestyle="--")
    interval = annotations[sid]["review_frames"]
    if interval:
        ax.axvspan(t[interval[0]], t[interval[1]], color="gray", alpha=0.2, label="Post-hoc review window")
    ax.set(title=sid, xlabel="Timestamp (seconds)", ylabel="Gripper value (raw units)")
    ax.legend(fontsize=7)
fig.suptitle("Training samples: gripper command and observed state", fontsize=13)
fig.tight_layout()
plt.show()
display(pd.DataFrame(offsets))
print("출처: 로컬 학습 영상 및 Parquet. 정규화·평활화하지 않은 값. 동일 timestamp는 센서 동기화 정확도의 보증이 아닙니다.")


## 5. 왜 task 문장을 정답으로 쓰면 안 되는가

동일 환경 80개 에피소드에 네 가지 수집 조건이 있습니다. 수집 조건과 실제 최종 성공 여부를 혼동하면 안 됩니다. 파일명·task_index도 조건과 연관될 수 있어 모델 입력에서 제외합니다.


In [ ]:
episode_meta = ROOT / "data/raw/data/train/sixpigs1/so100_stack_cube_error/meta/episodes.jsonl"
collection_conditions = [task for line in episode_meta.read_text().splitlines() if line.strip() for task in json.loads(line).get("tasks", [])]
display(pd.Series(collection_conditions).value_counts().rename_axis("collection_condition").to_frame("episodes"))
print("각 조건 20개는 수집 메타데이터 수이며, 성공·실패 판정 수가 아닙니다.")


## 다음 단계

1. 쌓기 80개 전체에 대해 사건 / 발생 구간 / 최종 결과 / 판단 불가 사유를 라벨링하고 사람이 검토합니다.
2. 이미 본 6개는 개발용으로 표시하고, 나머지를 검증과 잠금 테스트로 에피소드 단위 분리합니다. 중복·연속 촬영 영향을 확인합니다.
3. 종결 이미지 기준 결과 확인, 영상 기반 사건 확인, 영상+행동 기반 확인을 비교합니다. 조기 탐지에는 미래 정보를 쓰지 않습니다.
4. 오탐 수, 실패 재현율, 탐지 지연, 판단 유보율을 평가합니다. 현재는 모델 학습 전입니다.

현 단계 결론: **동일 환경에서의 소규모 검증은 진행 가능. 현업 일반화와 무알람 실패 주장은 아직 미검증.**
